# 01 — Bronze Extract | RHC Training Analytics

Este notebook implementa a camada **Bronze** do projeto RHC Training Analytics a partir de arquivos CSV exportados do Supabase e armazenados no Google Drive.

Fluxo: `Supabase → CSV RAW → validação → Parquet Bronze`.

Na Bronze não aplicamos regras de negócio: preservamos os dados da origem e adicionamos auditoria, rastreabilidade e validações técnicas.


## 1. Montar o Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Imports e configuração da execução


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import hashlib
import pandas as pd

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

RUN_TS = datetime.now(timezone.utc)
RUN_ID = RUN_TS.strftime('%Y%m%dT%H%M%SZ')
EXTRACT_DATE = RUN_TS.strftime('%Y-%m-%d')

BASE_DIR = Path('/content/drive/MyDrive/rhc-training-analytics/data')
RAW_DIR = BASE_DIR / 'raw'
BRONZE_DIR = BASE_DIR / 'bronze' / f'extract_date={EXTRACT_DATE}' / f'run_id={RUN_ID}'
BRONZE_DIR.mkdir(parents=True, exist_ok=True)

print('RAW:', RAW_DIR)
print('BRONZE:', BRONZE_DIR)
print('RUN_ID:', RUN_ID)


## 3. Manifesto dos arquivos esperados


In [ ]:
SOURCE_TABLES = [
    'profiles',
    'body_measurements',
    'workout_sessions',
    'workout_exercises',
    'exercise_catalog',
    'exercise_records',
    'training_programs',
    'program_phases',
    'program_sessions',
    'program_exercises',
    'program_enrollments',
    'program_exercise_exposures',
]

EXPECTED_FILES = [f'{table}.csv' for table in SOURCE_TABLES]
EXPECTED_FILES


## 4. Validar arquivos RAW no Drive


In [ ]:
available_files = sorted([p.name for p in RAW_DIR.glob('*.csv')])
missing_files = sorted(set(EXPECTED_FILES) - set(available_files))
extra_files = sorted(set(available_files) - set(EXPECTED_FILES))

print(f'Arquivos encontrados: {len(available_files)}')
print('Ausentes:', missing_files or 'nenhum')
print('Extras:', extra_files or 'nenhum')

if missing_files:
    raise FileNotFoundError(f'Arquivos RAW ausentes: {missing_files}')


## 5. Funções auxiliares

Cada CSV é lido sem transformação de negócio e persistido em Parquet. Também registramos checksum SHA-256 para rastreabilidade do arquivo de origem.


In [ ]:
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def read_raw_csv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, low_memory=False)


def extract_to_bronze(table: str) -> dict:
    source_path = RAW_DIR / f'{table}.csv'
    target_path = BRONZE_DIR / f'{table}.parquet'
    started_at = datetime.now(timezone.utc)

    df = read_raw_csv(source_path)
    df.to_parquet(target_path, index=False, engine='pyarrow')

    finished_at = datetime.now(timezone.utc)
    return {
        'table': table,
        'source_file': source_path.name,
        'rows': len(df),
        'columns': len(df.columns),
        'source_size_bytes': source_path.stat().st_size,
        'source_sha256': sha256_file(source_path),
        'bronze_file': target_path.name,
        'bronze_size_bytes': target_path.stat().st_size,
        'started_at_utc': started_at.isoformat(),
        'finished_at_utc': finished_at.isoformat(),
        'status': 'success',
    }


## 6. Executar carga Bronze


In [ ]:
results = []

for table in SOURCE_TABLES:
    print(f'Processando {table}...', end=' ')
    try:
        result = extract_to_bronze(table)
        results.append(result)
        print(f"OK - {result['rows']} linhas")
    except Exception as exc:
        results.append({
            'table': table,
            'source_file': f'{table}.csv',
            'rows': None,
            'columns': None,
            'source_size_bytes': None,
            'source_sha256': None,
            'bronze_file': None,
            'bronze_size_bytes': None,
            'started_at_utc': None,
            'finished_at_utc': datetime.now(timezone.utc).isoformat(),
            'status': 'error',
            'error': str(exc),
        })
        print(f'ERRO - {exc}')

audit_df = pd.DataFrame(results)
audit_df


## 7. Profiling técnico da origem


In [ ]:
profile_records = []

for table in SOURCE_TABLES:
    df = read_raw_csv(RAW_DIR / f'{table}.csv')
    for column in df.columns:
        s = df[column]
        profile_records.append({
            'table': table,
            'column': column,
            'pandas_dtype': str(s.dtype),
            'rows': len(df),
            'null_count': int(s.isna().sum()),
            'null_pct': round(float(s.isna().mean() * 100), 2),
            'distinct_count': int(s.nunique(dropna=True)),
        })

profile_df = pd.DataFrame(profile_records)
profile_df.head(30)


## 8. Validar CSV RAW × Parquet Bronze


In [ ]:
validation_records = []

for table in SOURCE_TABLES:
    raw_df = read_raw_csv(RAW_DIR / f'{table}.csv')
    bronze_df = pd.read_parquet(BRONZE_DIR / f'{table}.parquet')

    validation_records.append({
        'table': table,
        'raw_rows': len(raw_df),
        'bronze_rows': len(bronze_df),
        'raw_columns': len(raw_df.columns),
        'bronze_columns': len(bronze_df.columns),
        'row_count_match': len(raw_df) == len(bronze_df),
        'column_count_match': len(raw_df.columns) == len(bronze_df.columns),
        'column_names_match': list(raw_df.columns) == list(bronze_df.columns),
    })

validation_df = pd.DataFrame(validation_records)
validation_df


## 9. Persistir auditoria e manifesto


In [ ]:
audit_df.to_parquet(BRONZE_DIR / '_extraction_audit.parquet', index=False)
profile_df.to_parquet(BRONZE_DIR / '_raw_profile.parquet', index=False)
validation_df.to_parquet(BRONZE_DIR / '_raw_bronze_validation.parquet', index=False)

manifest = {
    'project': 'rhc-training-analytics',
    'layer': 'bronze',
    'source': 'Supabase CSV export',
    'run_id': RUN_ID,
    'run_timestamp_utc': RUN_TS.isoformat(),
    'raw_directory': str(RAW_DIR),
    'bronze_directory': str(BRONZE_DIR),
    'tables_expected': len(SOURCE_TABLES),
    'tables_success': int((audit_df['status'] == 'success').sum()),
    'tables': SOURCE_TABLES,
    'output_format': 'parquet',
}

with (BRONZE_DIR / '_manifest.json').open('w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

manifest


## 10. Critério de sucesso da Bronze


In [ ]:
errors = audit_df[audit_df['status'] != 'success']
invalid = validation_df[
    ~(validation_df['row_count_match'] & validation_df['column_count_match'] & validation_df['column_names_match'])
]

assert errors.empty, f"Falha na carga de: {errors['table'].tolist()}"
assert invalid.empty, f"Divergência RAW × Bronze em: {invalid['table'].tolist()}"

print(f'✅ Bronze concluída com sucesso: {len(SOURCE_TABLES)} tabelas processadas.')
print(f'Arquivos gerados em: {BRONZE_DIR}')
print('Próxima etapa: 02_silver_transform.ipynb')
